# VietTDR — quy trình huấn luyện đầy đủ trên Kaggle

## Chuẩn bị (làm một lần)

Tạo **hai** Kaggle Dataset (Datasets → New Dataset → Private → Create):

| Tên dataset | File upload | Kích thước |
|---|---|---|
| `viettdr-data` | `vintext_words.zip` | 155 MB |
| `viettdr-code` | `VietTDR.zip` | 4.9 MB (đã kèm 21 font) |

Sau đó: **Code → New Notebook → File → Import Notebook** (file này) →
**Add Input** cả hai dataset → **Session options → Accelerator = GPU T4 x2**.

Không cần bật Internet — font đã nằm sẵn trong `VietTDR.zip`.

Khi nâng cấp mã nguồn: tạo dataset mới tên có chữ `code` (ví dụ
`viettdr-code-v4`), Add Input, chạy lại cell 2. Cell 2 luôn ưu tiên dataset
mới nhất có chữ `code`, và **không đụng tới 155 MB dữ liệu**.

## Trình tự chạy

| Cell | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra GPU | 5 giây |
| 2 | Nạp code + copy dữ liệu về ổ local | 2 phút |
| 3 | Kiểm thử module phân rã ký tự | 10 giây |
| 4 | Sinh 200k ảnh tổng hợp (cân bằng thanh điệu) | ~20 phút |
| 5 | Huấn luyện model đầy đủ (2 giai đoạn) | ~45 phút |
| 6 | Ba thí nghiệm đối chứng | ~2.2 giờ |
| 7 | Đánh giá cả 4 trên tập test | ~10 phút |
| 8 | Đóng gói kết quả để tải về | 1 phút |

Tổng ~3.5 giờ, nằm gọn trong giới hạn 12 giờ một phiên.

**Nên dùng Save Version → Save & Run All** để chạy nền, không phụ thuộc
trình duyệt. Chạy tương tác thì phải giữ tab mở và tắt chế độ ngủ của máy.

## Bốn thí nghiệm

| Run | Cờ | Cô lập điểm mới |
|---|---|---|
| `full` | (mặc định) | — |
| `flat` | `--no-decompose` | ① phân rã ký tự ba thành phần |
| `no_toneprior` | `--no-tone-prior` | ② tiên nghiệm vị trí thanh điệu |
| `no_comploss` | `--lam-c 0.0` | ③ ràng buộc tổ hợp hợp lệ |

Cả bốn dùng **chung một bộ siêu tham số và cùng quy trình 2 giai đoạn**
(pretrain tổng hợp → tinh chỉnh VinText) để bảng so sánh công bằng.

## Đọc log thế nào

```
ep  12 | loss 2.41 (b 1.62 m 0.42 t 0.44 c 0.11) | word 0.31 1-NED 0.78 heavy 0.19 | lr 6.9e-04 | 42s
```

- **1-NED** = độ chính xác mức ký tự — tăng mượt ngay từ đầu, đây là chỉ
  số nên theo dõi.
- **word** = đúng nguyên từ — đường cong chữ S, nằm sát 0 khá lâu rồi mới
  bật lên. Thấp ở epoch đầu là bình thường.
- **heavy** = độ chính xác trên tập từ nhiều dấu (>2 ký tự mang dấu) — chỉ
  số quan trọng nhất cho bài báo.
- Epoch không đánh giá hiển thị `nan` (do `--eval-every 3`), đó là chủ ý.

## Nếu phiên bị ngắt giữa chừng

Chạy lại cell 1, 2 rồi thêm `--resume runs/<tên>/last.pth` vào lệnh train
đang dở. Checkpoint nằm trong `/kaggle/working` nên còn nguyên.

In [ ]:
# Cell 1 — moi truong
import torch, os, multiprocessing
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '*** KHONG CO GPU - vao Session options bat Accelerator ***')
print('CPU   :', multiprocessing.cpu_count(), 'cores')
print('torch :', torch.__version__)
IS_KAGGLE = os.path.exists('/kaggle')
NPROC = max(2, multiprocessing.cpu_count())
print('nen tang:', 'Kaggle' if IS_KAGGLE else 'Colab/khac')

In [ ]:
# Cell 2 — nap code + du lieu
# Uu tien dataset co chu 'code' trong ten (ban moi nhat).
# Du lieu duoc COPY sang o local: doc 41k anh nho truc tiep tu /kaggle/input
# cham hon nhieu va se la nut co chai cua moi epoch.
import os, glob, shutil, time

if IS_KAGGLE:
    SRC, WORK = '/kaggle/input', '/kaggle/working/run'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC, WORK = '/content/drive/MyDrive', '/content/run'
os.makedirs(WORK, exist_ok=True)

def find_all(root, name):
    return sorted(glob.glob(f'{root}/**/{name}', recursive=True))

def pick(hits, prefer='code'):
    if not hits:
        return None
    for h in hits:
        if prefer in h.lower():
            return h
    return hits[0]

# zip con nguyen (Colab/Drive) -> giai nen truoc
for z, dest in [(pick(find_all(SRC, 'VietTDR.zip')), WORK),
                (pick(find_all(SRC, 'vintext_words.zip'), 'vintext'),
                 f'{WORK}/data')]:
    if z:
        get_ipython().system(f'unzip -qo "{z}" -d "{dest}"')

# --- CODE ---
train_py = pick(find_all(SRC, 'train.py')) or pick(find_all(WORK, 'train.py'))
assert train_py, 'Khong tim thay train.py - da Add Input dataset code chua?'
code_root = os.path.dirname(train_py)
if os.path.abspath(code_root) != os.path.abspath(WORK):
    for item in ['viettdr', 'tools', 'tests', 'fonts', 'assets',
                 'train.py', 'eval.py', 'charset_vintext.txt']:
        s, d = os.path.join(code_root, item), os.path.join(WORK, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        elif os.path.isfile(s):
            shutil.copy(s, d)

# --- DATA: copy sang o local (chi lam 1 lan) ---
gt = pick(find_all(SRC, 'train_gt.jsonl'), 'vintext') or \
     pick(find_all(WORK, 'train_gt.jsonl'), 'vintext')
assert gt, 'Khong tim thay train_gt.jsonl - dataset du lieu da upload chua?'
data_dst = os.path.join(WORK, 'data', 'vintext_words')
if not os.path.exists(os.path.join(data_dst, 'train_gt.jsonl')):
    t0 = time.time()
    if os.path.lexists(data_dst):
        os.remove(data_dst) if os.path.islink(data_dst) else shutil.rmtree(data_dst)
    os.makedirs(os.path.dirname(data_dst), exist_ok=True)
    shutil.copytree(os.path.dirname(gt), data_dst)
    print(f'copy du lieu -> o local: {time.time() - t0:.0f}s')

os.chdir(WORK)
print('code :', code_root)
print('data :', data_dst)
print('font :', len(glob.glob(f'{WORK}/fonts/*.ttf')), 'file')

# xac nhan ban code day du — PHAI du 6 dong [OK], co [THIEU] la nap nham ban cu
src = open('train.py', encoding='utf-8').read()
checks = {'--no-decompose': src, '--init-from': src, 'GradScaler': src,
          '--val-subset': src, '1-NED': src,
          'assets/general_dict.txt': ('x' if os.path.exists('assets/general_dict.txt') else '')}
for flag, hay in checks.items():
    print(('  [OK]    ' if (flag in hay or hay == 'x') else '  [THIEU] ') + flag)
print()
get_ipython().system('ls data/vintext_words/')

In [ ]:
# Cell 3 — kiem thu module phan ra ky tu (phai in ALL TESTS PASSED)
!python tests/test_vietchar.py

In [ ]:
# Cell 4 — SINH DU LIEU TONG HOP (~20 phut cho 200k anh tren 4 core)
#
# Bat buoc: VinText chi co 25k anh tu, khong du de train tu dau. Moi phuong
# phap hien dai (PARSeq/ABINet/SVTR/VietOCR) deu pretrain tren anh tong hop.
#
# --tone-balance: thanh nga trong VinText chi chiem 0.8% so ky tu; sau can
# bang len ~4%. Chi bieu dien phan ra moi cho phep can bang theo thanh dieu.
N_SYNTH = 200000

import os
# general_dict.txt (CO dau) da duoc dong goi san trong zip code (assets/).
# KHONG dung vn_dictionary.txt - file do thuan ASCII, sinh ra du lieu se
# khong co thanh dieu nao.
corpus = ['data/vintext_words/train_gt.jsonl']
if os.path.exists('assets/general_dict.txt'):
    corpus.append('assets/general_dict.txt')
else:
    print('CANH BAO: thieu assets/general_dict.txt - corpus se ngheo hon')
corpus = ' '.join(corpus)
print('corpus:', corpus)

!python tools/gen_synth.py --out data/synth --count {N_SYNTH} \
    --fonts fonts --corpus {corpus} --workers {NPROC} \
    --bg-cache 40 --tone-balance

# kiem tra nhanh: bang thanh dieu phai du 6 nhom (level/acute/grave/hook/
# tilde/dot). Neu chi co 'level' -> corpus sai, DUNG LAI bao loi.
import json
from collections import Counter
import sys
sys.path.insert(0, '.')
from viettdr.vietchar import decompose_char
tones = Counter()
with open('data/synth/synth_gt.jsonl', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 20000:
            break
        for ch in json.loads(line)['text']:
            tones[decompose_char(ch)[2]] += 1
n = sum(tones.values())
print({t: f'{100*v/n:.1f}%' for t, v in sorted(tones.items())})
assert len(tones) == 6, 'THIEU THANH DIEU trong du lieu tong hop!'
print('[OK] du lieu tong hop du 6 thanh dieu')

In [ ]:
# Cell 5 — MODEL DAY DU, hai giai doan (~45 phut)
#   giai doan 1: pretrain tren synth + VinText
#   giai doan 2: tinh chinh chi tren VinText (--init-from nap trong so,
#                reset optimizer va lich learning rate)
import os

PRE = (f'--data data/vintext_words --charset charset_vintext.txt '
       f'--synth-jsonl data/synth/synth_gt.jsonl --synth-dir data/synth '
       f'--epochs 6 --bs 192 --workers {NPROC} --eval-every 2 --val-subset 2000')
FT = (f'--data data/vintext_words --charset charset_vintext.txt '
      f'--epochs 60 --bs 192 --workers {NPROC} --eval-every 3 '
      f'--val-subset 2000 --lr 2e-4')

def ckpt(run):
    """best.pth neu co, khong thi last.pth — de FT khong bao gio crash."""
    for fn in ['best.pth', 'last.pth']:
        p = f'runs/{run}/{fn}'
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'runs/{run} chua co checkpoint nao')

!python train.py {PRE} --out runs/pre_full
ck = ckpt('pre_full')
!python train.py {FT} --out runs/full --init-from {ck}

In [ ]:
# Cell 6 — BA THI NGHIEM DOI CHUNG (~2.2 gio)
# Cung quy trinh, cung sieu tham so, chi khac dung MOT co -> chenh lech
# la dong gop thuan cua diem moi tuong ung.
# (Dinh nghia lai PRE/FT/ckpt de cell nay tu chay duoc sau khi phien bi ngat.)
import os

PRE = (f'--data data/vintext_words --charset charset_vintext.txt '
       f'--synth-jsonl data/synth/synth_gt.jsonl --synth-dir data/synth '
       f'--epochs 6 --bs 192 --workers {NPROC} --eval-every 2 --val-subset 2000')
FT = (f'--data data/vintext_words --charset charset_vintext.txt '
      f'--epochs 60 --bs 192 --workers {NPROC} --eval-every 3 '
      f'--val-subset 2000 --lr 2e-4')

def ckpt(run):
    for fn in ['best.pth', 'last.pth']:
        p = f'runs/{run}/{fn}'
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'runs/{run} chua co checkpoint nao')

for name, flag in [('flat',         '--no-decompose'),    # (1) phan ra
                   ('no_toneprior', '--no-tone-prior'),   # (2) tone prior
                   ('no_comploss',  '--lam-c 0.0')]:      # (3) rang buoc
    print('#' * 22, name, '#' * 22, flush=True)
    !python train.py {PRE} --out runs/pre_{name} {flag}
    ck = ckpt('pre_' + name)
    !python train.py {FT} --out runs/{name} {flag} --init-from {ck}

In [ ]:
# Cell 7 — DANH GIA tren toan bo 9789 anh test
import os
for run in ['full', 'flat', 'no_toneprior', 'no_comploss']:
    ck = f'runs/{run}/best.pth'
    if os.path.exists(ck):
        print('=' * 24, run, '=' * 24, flush=True)
        !python eval.py --ckpt {ck} --data data/vintext_words --split test --charset charset_vintext.txt
    else:
        print(f'[bo qua] {run}: chua co checkpoint')

In [ ]:
# Cell 8 — dong goi ket qua
# Checkpoint day du chua ca trang thai AdamW (~240MB); chi giu trong so
# model (~80MB) cho nhe khi tai ve.
import shutil, os, torch, json

os.makedirs('/tmp/results', exist_ok=True)
summary = {}
for r in ['full', 'flat', 'no_toneprior', 'no_comploss']:
    dst = f'/tmp/results/{r}'
    os.makedirs(dst, exist_ok=True)
    for f in ['log.csv', 'eval_test.json']:
        if os.path.exists(f'runs/{r}/{f}'):
            shutil.copy(f'runs/{r}/{f}', dst)
    if os.path.exists(f'runs/{r}/eval_test.json'):
        s = json.load(open(f'runs/{r}/eval_test.json', encoding='utf-8'))['summary']
        summary[r] = {k: round(s[k], 4) for k in
                      ['word_acc', 'word_acc_ci', 'one_minus_ned', 'heavy_acc']}
    ck = f'runs/{r}/best.pth'
    if os.path.exists(ck):
        c = torch.load(ck, map_location='cpu', weights_only=False)
        torch.save({'model': c['model'], 'args': c['args'],
                    'epoch': c['epoch'], 'best': c['best']}, f'{dst}/best_slim.pth')

print(json.dumps(summary, indent=1, ensure_ascii=False))
json.dump(summary, open('/tmp/results/summary.json', 'w'), indent=1)

dest = '/kaggle/working' if IS_KAGGLE else SRC
shutil.make_archive(f'{dest}/results_viettdr', 'zip', '/tmp/results')
print(f'\nXONG -> {dest}/results_viettdr.zip '
      f'({os.path.getsize(dest + "/results_viettdr.zip")/1e6:.0f} MB)')
print('Tai ve o panel Output ben phai (hoac tab Output cua Version).')